# Case 1: DNN and Interpretation

## Setup

Imagine you work on an analytics team at a bank/fintech. Your team would like to use machine learning to predict which customers are likely to default next month. The predictions could then be used to inform interventions (e.g., outreach, changing credit terms, or tightening limits). Your team is aware that deploying such algorithms may also create unfair or legally risky outcomes across demographic groups.

This case focuses on a simple [publicly available dataset](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) for simplicity and tractability.

The data are 30000 observations on 23 features:

### Variable definitions

- `ID`: Client identifier.

- `LIMIT_BAL`: Amount of the given credit (NT dollars). Includes both individual consumer credit and family (supplementary) credit.

- `SEX`: Gender (`1` = male; `2` = female).

- `EDUCATION`: Education (`1` = graduate school; `2` = university; `3` = high school; `4` = others).

- `MARRIAGE`: Marital status (`1` = married; `2` = single; `3` = others).

- `AGE`: Age (years).

### Repayment status history

These variables record repayment status by month, with the following coding:
- `-2` = no consumption
- `-1` = pay duly/pay on time
- `0` = use of revolving credit
- `1` = payment delay for one month
- `2` = payment delay for two months
- …
- `8` = payment delay for eight months
- `9` = payment delay for nine months and above

Month mapping:
- `PAY_0`: Repayment status this month
- `PAY_2`: Repayment status last month
- `PAY_3`: Repayment status two months ago
- `PAY_4`: Repayment status three months ago
- `PAY_5`: Repayment status four months ago
- `PAY_6`: Repayment status five months ago

### Bill statement amounts

Month mapping:
- `BILL_AMT1`: Bill statement amount this month
- `BILL_AMT2`: Bill statement amount last month
- `BILL_AMT3`: Bill statement amount two months ago
- `BILL_AMT4`: Bill statement amount three months ago
- `BILL_AMT5`: Bill statement amount four months ago
- `BILL_AMT6`: Bill statement amount five months ago

### Previous payment amounts

Month mapping:
- `PAY_AMT1`: Amount paid this month
- `PAY_AMT2`: Amount paid last month
- `PAY_AMT3`: Amount paid two months ago
- `PAY_AMT4`: Amount paid three months ago
- `PAY_AMT5`: Amount paid four months ago
- `PAY_AMT6`: Amount paid five months ago

### Outcome

- `default payment next month`: Indicator for whether the client defaults on payment in the next month (binary outcome).

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

!pip install fairlearn
from fairlearn.metrics import (
    MetricFrame,
    selection_rate, true_positive_rate, false_positive_rate,
    demographic_parity_difference, demographic_parity_ratio,
    equalized_odds_difference, equalized_odds_ratio
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
import numpy as np
import pandas as pd

# Get data from dropbox
# Import data from dropbox
!wget -O defaultCCclients.xls https://www.dropbox.com/scl/fi/k87irqufuzjfe3psv9pg1/defaultCCclients.xls?rlkey=e1w5ti23cwk719qbd5ye4w134&dl=0

# Read the excel file, skipping the first row (metadata) and using the second row (index 1) as header.
credit_df = pd.read_excel("defaultCCclients.xls", header=1)

# Extract features (X) and target (y).
# Drop the 'ID' column and the 'default payment next month' column from the features.
X = credit_df.drop(columns=['ID', 'default payment next month']).copy()
# The target variable is 'default payment next month'.
y = credit_df['default payment next month'].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns:", list(X.columns)[:10], "...")
print("y value counts:", pd.Series(y).value_counts().head())

In [ ]:
X.head()

### Sensitive Attribute

We're going to set things up treating `SEX` as a sensitive attribute.

In [ ]:
A_sex = X["SEX"]

In [ ]:
# Train/test/validation splitting
X_trainval, X_test, y_trainval, y_test, A_trainval, A_test = train_test_split(
    X, y, A_sex, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

X_train, X_val, y_train, y_val, A_train, A_val = train_test_split(
    X_trainval, y_trainval, A_trainval, test_size=0.25, random_state=RANDOM_SEED, stratify=y_trainval
)  # => 60% train, 20% val, 20% test

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Default rate (train/val/test):", y_train.mean(), y_val.mean(), y_test.mean())

### Setting up to estimate our baseline DNN

We're going to use PyTorch here. We need to do some setup before estimating our baseline model.

In [ ]:
# Standardizing data is important for NN training
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Some preliminary setup before defining the NN architecture
class TabDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.float32).view(-1, 1)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_train_s, y_train.to_numpy())
val_ds   = TabDataset(X_val_s, y_val.to_numpy())
test_ds  = TabDataset(X_test_s, y_test.to_numpy())

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False)

**Question 1.** Briefly explain what the above code block is doing. Do you think the standardization makes sense? Is there a more natural way to treat some of the variables?

(HINT: Look at the variable definitions in the first code-block.)

---

We're now going to define the architecture for our baseline model.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)  # logits
        )
    def forward(self, x):
        return self.net(x)

model = MLP(d_in=X_train_s.shape[1])

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

---

**Question 2.** Briefly explain the architecture of the baseline model. Feel free to use a schematic.


---


We're now going to train the model. Here we're using early stopping.

In [ ]:
def predict_proba_torch(model, loader):
    model.eval()
    ps = []
    ys = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            p = torch.sigmoid(logits)
            ps.append(p.numpy().ravel())
            ys.append(yb.numpy().ravel())
    return np.concatenate(ys), np.concatenate(ps)

best_state = None
best_val_auc = -np.inf
patience = 3
pat = 0

for epoch in range(1, 21):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    yv, pv = predict_proba_torch(model, val_loader)
    val_auc = roc_auc_score(yv, pv)
    print(f"Epoch {epoch:02d} | val AUC = {val_auc:.4f}")

    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

model.load_state_dict(best_state)

In [ ]:
yt, pt = predict_proba_torch(model, test_loader)
print("DNN AUC (test):", roc_auc_score(yt, pt))

### Costs

Let's just pretend that we are willing to trade off mistakes by treating a false negative as being 10 times more costly than a false positive. We'll benchmark the cost of a false positive to 1, and thus have the cost of a false negative as 10.

---
**Question 3.** Provide an intuitive discussion of possible motivations for this cost structure. Explain why this reasoning might make sense in an environment where you did not have firm monetary values for false positives and false negatives. Explain how you could use conversations with stakeholders/insiders to adjust these values. Part of this explanation should include a clear definition of what a false positive and a false negative are in this example.

---

Given these costs, let's evaluate our model on the basis of expected cost.



In [ ]:
# Helper functions to evaluate expected costs

def expected_cost(y_true, y_hat, cost_fn=10.0, cost_fp=1.0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    return cost_fn*fn + cost_fp*fp

def choose_threshold_by_cost(y_true, p, cost_fn=10.0, cost_fp=1.0, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    best = None
    for t in grid:
        y_hat = (p >= t).astype(int)
        c = expected_cost(y_true, y_hat, cost_fn=cost_fn, cost_fp=cost_fp)
        if (best is None) or (c < best["cost"]):
            best = {"t": t, "cost": c}
    return best


In [ ]:
best_dnn = choose_threshold_by_cost(y_val, predict_proba_torch(model, val_loader)[1], cost_fn=10.0, cost_fp=1.0)
t_dnn = best_dnn["t"]

yhat_test_dnn = (pt >= t_dnn).astype(int)

print("Chosen threshold (DNN):", t_dnn)
print("Test confusion matrix (DNN):\n", confusion_matrix(yt, yhat_test_dnn))
print("Test expected cost (DNN):", expected_cost(yt, yhat_test_dnn))

**Question 4.** Interpret the output above (`Chosen threshold`, `Test confusion matrix`, and `Test expected cost`).

---


While it is hard (impossible given human time/resource constraints?) to understand exactly how DNN predictions are made, we often want to at least try to get a sense of the underlying model.

One approach we discussed in class is permutation feature importance. Here, we're going to apply it to our NN prediction model.

In [ ]:
feature_names = list(X.columns)

def dnn_predict_proba_from_numpy(Xs_np):
    # Xs_np must already be scaled
    loader = DataLoader(TabDataset(Xs_np, np.zeros(len(Xs_np))), batch_size=1024, shuffle=False)
    model.eval()
    ps = []
    with torch.no_grad():
        for xb, _ in loader:
            ps.append(torch.sigmoid(model(xb)).numpy().ravel())
    return np.concatenate(ps)

def permutation_importance_auc(model_predict_proba_fn, Xs, y_true, n_repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y_true, model_predict_proba_fn(Xs))
    importances = []
    for j in range(Xs.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = Xs.copy()
            rng.shuffle(Xp[:, j])
            auc_p = roc_auc_score(y_true, model_predict_proba_fn(Xp))
            drops.append(base - auc_p)
        importances.append(np.mean(drops))
    return base, np.array(importances)

base_auc, imps = permutation_importance_auc(dnn_predict_proba_from_numpy, X_test_s, y_test.to_numpy(), n_repeats=2, seed=RANDOM_SEED)

imp_df = pd.DataFrame({"feature": feature_names, "auc_drop": imps}).sort_values("auc_drop", ascending=False)
imp_df.head(15)

**Question 5.** Using the permutation importance table, identify the top 5 drivers of the DNN's predictive performance. For each of the identified variables, give a business interpretation and be sure to comment on what the variable might be proxying for. Finally, identify at least one feature that could raise concerns about proxy discrimination or sensitive information leakage (even if you remove `SEX`).

---


### Fairness assessment

There are many different ways to measure and assess fairness in algorithms. Let's look at an assessment using our baseline model.

In [ ]:
# Use the DNN threshold chosen earlier
A_test_series = pd.Series(A_test).reset_index(drop=True)
y_test_series = pd.Series(y_test).reset_index(drop=True)

# Align arrays
p_test_dnn = pd.Series(pt).reset_index(drop=True)
yhat_test_dnn = (p_test_dnn >= t_dnn).astype(int)

metrics = {
    "selection_rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
}

mf = MetricFrame(
    metrics=metrics,
    y_true=y_test_series,
    y_pred=yhat_test_dnn,
    sensitive_features=A_test_series
)

print(mf.by_group)

# ---- Summary fairness metrics (demographic parity + equalized odds) ----
dp_diff  = demographic_parity_difference(y_test_series, yhat_test_dnn, sensitive_features=A_test_series)
dp_ratio = demographic_parity_ratio(y_test_series, yhat_test_dnn, sensitive_features=A_test_series)

eo_diff  = equalized_odds_difference(y_test_series, yhat_test_dnn, sensitive_features=A_test_series)
eo_ratio = equalized_odds_ratio(y_test_series, yhat_test_dnn, sensitive_features=A_test_series)

print("\nFairness summary metrics (across groups):")
print(f"Demographic parity difference: {dp_diff:.4f}")
print(f"Demographic parity ratio:  {dp_ratio:.4f}")
print(f"Equalized odds difference:  {eo_diff:.4f}")
print(f"Equalized odds ratio:  {eo_ratio:.4f}")

**Question 6.** Interpret the results from the fairness audit.

What is selection rate? Does it differ by group? What operational outcomes does it correspond to?

Compare TPR and FPR across groups. Is the TPR or FPR contrast likely more important in this context? Explain.

We also report demographic parity and equalized odds. Explain what these two objects measure. (You will need to "look it up.") Be sure that the explanation conveys how they differ from each other. Does either seem more appropriate in the present context? Explain.

---

### "Improving Fairness"

A simple and interpretable way to "improve fairness" is to allow different decision thresholds by group chosen to specifically target a fairness criterion.



In [ ]:
def threshold_to_hit_tpr(y_true, p, target_tpr):
    # search over thresholds; pick one with closest TPR
    grid = np.linspace(0.01, 0.99, 99)
    best = None
    for t in grid:
        yhat = (p >= t).astype(int)
        tpr = true_positive_rate(y_true, yhat)
        if best is None or abs(tpr - target_tpr) < best["gap"]:
            best = {"t": t, "tpr": tpr, "gap": abs(tpr - target_tpr)}
    return best

# Choose a target TPR = overall TPR of the original DNN threshold
overall_tpr = true_positive_rate(y_test_series, yhat_test_dnn)
overall_tpr

In [ ]:
# Compute group-specific thresholds
groups = sorted(A_test_series.unique())
thresholds = {}

for g in groups:
    idx = (A_test_series == g).to_numpy()
    best_g = threshold_to_hit_tpr(y_test_series[idx].to_numpy(), p_test_dnn[idx].to_numpy(), target_tpr=overall_tpr)
    thresholds[g] = best_g["t"]

thresholds

In [ ]:
# Apply group thresholds and evaluate
yhat_group = np.zeros(len(y_test_series), dtype=int)
for g in groups:
    idx = (A_test_series == g).to_numpy()
    yhat_group[idx] = (p_test_dnn[idx].to_numpy() >= thresholds[g]).astype(int)

mf_post = MetricFrame(
    metrics=metrics,
    y_true=y_test_series,
    y_pred=yhat_group,
    sensitive_features=A_test_series
)

print("Original DNN by-group:\n", mf.by_group)
print("\nPost-processed (TPR-matched) by-group:\n", mf_post.by_group)
print("\nExpected cost (original):", expected_cost(y_test_series, yhat_test_dnn))
print("Expected cost (post-processed):", expected_cost(y_test_series, yhat_group))

**Question 7.** Evaluate the results of the post-processed decision rule. Comment on changes to TPR, FPR, and the selection rate. How does the expected cost metric change? Explain why these changes make intuitive sense. Would you recommend deploying the original or post-processed rule? Explain.

What are the pros and cons to using differing decision thresholds? Do you believe doing so could lead to downstream issues? Present ideas for other ways to "improve" fairness.

## Main Question Prompt

**Question 8.** Your goal is now to build the most effective prediction rule for `default payment next month` that you can (using DNNs), while being mindful of fairness considerations. (Note that a firm may be interested in other dimensions of fairness beyond `SEX`.) There is no single correct approach. A strong solution will require making choices, justifying them, and testing alternatives.

Clearly state your objective in this setting. You may use the same costs defined above (but feel free to change them if you think it makes sense). Clearly state the way(s) you will assess predictive performance and fairness of predictions. Explain why you focus on your chosen measures.

For this problem, focus on DNN classification rules. (I.e., don't bother with other methods we previously discussed.) Consider different architectures and/or tuning choices. Explain why you make the choices you do. If you choose to make transformations to the variables outside of the standardization done in the problem set-up, be sure to document and justify any such transformations.

Choose a final model. Carefully justify why you chose that model given your criteria.

This part of the solution should be provided as a slide deck (and notes) suitable for a 10 minute presentation. Imagine your audience are business stakeholders who are motivated chiefly by profit, are wary of employing algorithms that they don't understand, and wish to avoid biased algorithms - to the extent possible - for legal, public relations, and ethical reasons.

# Case 2: Introduction to Convolutional Neural Networks

## Setup

Imagine you are part of an operations and risk team at a fintech that processes large volumes of customer paperwork (e.g., applications, claims, onboarding forms). A key step is extracting handwritten digits from scanned fields (e.g., account numbers, routing numbers, ID numbers). Today, the firm uses a fully manual process: a human reviewer reads each digit and enters it into a system.

Your team is considering deploying a machine learning system to auto-read easy digits and route uncertain digits to humans.

This case focuses on a simplified subproblem: recognizing a single handwritten digit (0-9) using a classic ML benchmark dataset (MNIST). However, the underlying business problem is relevant:

> **How much should we automate, and how do we control risk?**

---

## Business objective: profit + risk controls

For each digit image, your system can choose:

- **Auto-process**: model predicts a digit and the system accepts it.
- **Manual review**: a human labels the digit (assume the human label is correct).

We will evaluate your system relative to a baseline where everything is manually reviewed. Let's pretend the firm faces the following costs:

- **Manual review cost:** \$0.06 per digit routed to humans  
- **Error cost (incorrect auto-processed digit):** \$3.00 per digit

With these costs, we have, relative to the all-manual baseline:

- Auto-process & correct → saves \$0.06  
- Auto-process & incorrect → saves \$0.06 but costs \$3.00 → net -\$2.94  
- Manual review → 0 incremental value (same as baseline)

### Capacity constraint

Finally, let's imagine that the firm does not have resources to manually review every document. We'll model this with the constraint that you can manually review at most 25% of digits. That is, your system must satisfy:

- Manual review rate ≤ 25% (equivalently, auto-processing rate ≥ 75%).

---

To start, we're going to build a baseline logistic classifier.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

# Data are MNIST from Keras
from tensorflow.keras.datasets import mnist

# For the baseline classifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


## Baseline code: load MNIST and visualize digits

In [ ]:
# Load MNIST
# x_train, x_test: uint8 images with shape (n, 28, 28)
# y_train, y_test: labels 0..9
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test: ", x_test.shape,  x_test.dtype)
print("y_test: ", y_test.shape,  y_test.dtype)

# Show a few digits
rng = np.random.default_rng(0)
idx = rng.choice(len(x_train), size=12, replace=False)

plt.figure(figsize=(10, 4))
for i, j in enumerate(idx, start=1):
    plt.subplot(3, 4, i)
    plt.imshow(x_train[j], cmap="gray")
    plt.title(f"Label: {y_train[j]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# --- Baseline: Logistic regression (linear classifier) ---

# Preprocess: flatten images and scale to [0, 1]
X_train = x_train.reshape(len(x_train), -1).astype(np.float32) / 255.0
X_test  = x_test.reshape(len(x_test), -1).astype(np.float32) / 255.0

# OPTIONAL: to make experiments run fast on laptops, you can subsample the training set.
# (You can comment this out if you want to use all 60,000 training images.)
n_train = 15000
rng = np.random.default_rng(1)
sub_idx = rng.choice(len(X_train), size=n_train, replace=False)

X_train_sub = X_train[sub_idx]
y_train_sub = y_train[sub_idx]

# Logistic regression via SGD (one-vs-rest)
# - max_iter kept small on purpose to keep the baseline simple and fast
# - You are welcome to tune these later
logit = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    max_iter=10,
    tol=1e-3,
    random_state=0,
)

logit.fit(X_train_sub, y_train_sub)

# Predictions and accuracy
yhat_test = logit.predict(X_test)
baseline_test_acc = accuracy_score(y_test, yhat_test)
print(f"Baseline logistic test accuracy: {baseline_test_acc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, yhat_test, labels=np.arange(10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
disp.plot(values_format="d")
plt.title("Baseline (Logistic) Confusion Matrix")
plt.show()

# Show a few misclassified examples
mis = np.where(yhat_test != y_test)[0]
print("Number of test mistakes:", len(mis))

show_k = 12
mis_idx = rng.choice(mis, size=show_k, replace=False)

plt.figure(figsize=(10, 4))
for i, j in enumerate(mis_idx, start=1):
    plt.subplot(3, 4, i)
    plt.imshow(x_test[j], cmap="gray")
    plt.title(f"True {y_test[j]} / Pred {yhat_test[j]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# We'll also keep predicted probabilities for later triage analysis.
# (SGDClassifier exposes predict_proba after fitting with loss='log_loss'.)
baseline_test_proba = logit.predict_proba(X_test)  # shape (n_test, 10)
baseline_pmax = baseline_test_proba.max(axis=1)


## Setup Questions

**Question 1.** Our baseline classifier is multiclass logistic regression trained with stochastic gradient descent.

Explain how the classifier is using the image data. Specifically, comment on the step "# Preprocess: flatten images and scale to [0, 1]". How is this treating the images? What is it doing with the spatial structure in the image data? Is the degree of accuracy surprising? Explain.

(HINT: What do you think would happen if we did not give all images in the same orientation and scale?)

---

In [ ]:
# --- Baseline triage policy evaluation ---

# We already computed:
# - yhat_test (predicted class)
# - baseline_test_proba (class probabilities)
# - baseline_pmax (max predicted probability)

manual_cost = 0.06
error_cost = 3.00

def evaluate_policy(y_true, y_pred, pmax, tau, manual_cost=0.06, error_cost=3.00):
    """Return metrics for threshold-based triage policy."""
    auto = pmax >= tau
    manual = ~auto

    manual_rate = manual.mean()
    auto_rate = auto.mean()

    # Accuracy among auto-processed cases
    if auto.sum() > 0:
        auto_acc = (y_pred[auto] == y_true[auto]).mean()
        auto_err = 1 - auto_acc
    else:
        auto_acc = np.nan
        auto_err = np.nan

    # Incremental value relative to all-manual baseline:
    # - if auto & correct: +manual_cost
    # - if auto & incorrect: +manual_cost - error_cost
    correct_auto = auto & (y_pred == y_true)
    incorrect_auto = auto & (y_pred != y_true)

    inc_value = (
        correct_auto.mean() * manual_cost
        + incorrect_auto.mean() * (manual_cost - error_cost)
    )

    # Scale up to per 10,000 digits for readability
    inc_value_per_10k = inc_value * 10000

    return {
        "tau": tau,
        "manual_rate": manual_rate,
        "auto_rate": auto_rate,
        "auto_acc": auto_acc,
        "inc_value_per_10k": inc_value_per_10k,
    }

taus = np.linspace(0.50, 0.99, 50)
rows = [evaluate_policy(y_test, yhat_test, baseline_pmax, tau, manual_cost, error_cost) for tau in taus]

# Print best feasible tau (manual_rate <= 25%)
feasible = [r for r in rows if r["manual_rate"] <= 0.25]
best = max(feasible, key=lambda r: r["inc_value_per_10k"]) if feasible else None

if best is None:
    print("No thresholds satisfied the manual review constraint. (This would be bad!)")
else:
    print("Best feasible policy (baseline logistic):")
    for k, v in best.items():
        if k in {"tau"}:
            print(f"  {k:>16}: {v:.3f}")
        elif k.endswith("rate") or k.endswith("acc"):
            print(f"  {k:>16}: {v:.3%}")
        else:
            print(f"  {k:>16}: {v:,.2f}")

# Simple plot: incremental value vs manual rate
manual_rates = np.array([r["manual_rate"] for r in rows])
values_10k = np.array([r["inc_value_per_10k"] for r in rows])

plt.figure(figsize=(7,4))
plt.plot(manual_rates, values_10k)
plt.axvline(0.25, linestyle="--")
plt.xlabel("Manual review rate")
plt.ylabel("Incremental value per 10,000 digits ($)")
plt.title("Baseline Logistic: Value vs Manual Review Rate")
plt.show()


A classification model is not automatically an automation system. To deploy, you need an operating policy that decides what to do with each prediction.

Here, we consider a simple triage policy:

- Let `p_max` be the model's predicted probability for its chosen digit.
- If `p_max ≥ τ`, **auto-process** using the model prediction.
- If `p_max < τ`, route to **manual review**.

Using the cost model:

- Manual review cost = \$0.06 per digit sent to humans
- Incorrect auto-processed digit cost = \$3.00

---

**Question 2.** Explain what the code block above is doing and how it relates to the triage policy. Explain what the results produced capture. As part of this explanation, be sure to comment on the business relevance of this evaluation relative to just looking at accuracy. How does this relate to whether the firm should automate (using the baseline model)?

---

We are now going to explore the use of a **convolutional neural network (CNN)**. CNNs are a baseline neural network architecture that is useful for image classification tasks.

**Question 3.** Describe what a convolutional neural network (CNN) is doing in an image classification task in language suitable for intelligent, business-savvy people who are otherwise unfamiliar with machine learning/data-science.

(HINT: Any AI can answer this question.)

---

### A small CNN: baseline implementation + comparison to logistic regression

The code below provides a *starter* CNN so you can begin comparing the baseline logistic regression to a more up-to-date approach.

In [ ]:
# --- Small CNN starter model ---

import tensorflow as tf
from tensorflow.keras import layers, models

# Preprocess for CNN: add channel dimension and scale to [0, 1]
X_train_cnn = x_train.astype(np.float32) / 255.0
X_test_cnn  = x_test.astype(np.float32) / 255.0

X_train_cnn = X_train_cnn[..., np.newaxis]  # (n, 28, 28, 1)
X_test_cnn  = X_test_cnn[..., np.newaxis]

# OPTIONAL: keep CNN training fast by using a subset
n_train_cnn = 20000
rng = np.random.default_rng(2)
sub_idx_cnn = rng.choice(len(X_train_cnn), size=n_train_cnn, replace=False)

X_train_cnn_sub = X_train_cnn[sub_idx_cnn]
y_train_cnn_sub = y_train[sub_idx_cnn]

cnn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

history = cnn.fit(
    X_train_cnn_sub, y_train_cnn_sub,
    validation_split=0.1,
    epochs=3,
    batch_size=128,
    verbose=1
)

# CNN test accuracy
cnn_test_probs = cnn.predict(X_test_cnn, verbose=0)
cnn_test_pred = cnn_test_probs.argmax(axis=1)
cnn_test_acc = accuracy_score(y_test, cnn_test_pred)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")

# Compare to baseline logistic regression
print(f"Baseline logistic test accuracy: {baseline_test_acc:.4f}")
print(f"Accuracy improvement (CNN - baseline): {cnn_test_acc - baseline_test_acc:.4f}")

# Optional: confusion matrix for CNN
cm_cnn = confusion_matrix(y_test, cnn_test_pred, labels=np.arange(10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=np.arange(10))
disp.plot(values_format="d")
plt.title("CNN Confusion Matrix")
plt.show()


**Question 4.** Explain what the code block above is doing. What is the CNN architecture? (Feel free to use diagrams.) What tuning and regularization choices were made beyond the architecture? Relative to the baseline logistic regression, how is the image data being used? (Specifically, how is the spatial structure in the image handled in the CNN relative to the logistic regression above.)

## Main Question Prompt

Your goal is now to design the best end-to-end automation system you can for digit entry, balancing automation and risk under the capacity constraint.

There is no single correct approach. A strong solution will require making choices, justifying them, and testing alternatives. You should expect to go beyond what we covered directly in lecture (using group discussion, AI tools, and outside research as needed).

At a minimum, you should:

1. Develop and compare at least two CNN variants (architecture and/or training choices)
2. Choose a triage policy (threshold τ or an alternative)
3. Optimize expected business value subject to manual review rate ≤ 25%,
4. Provide a recommendation for deployment, including risk controls and a monitoring plan.

---

**Question 5.** Develop and compare a set of classification models. You should consider CNNs that differ meaningfully (e.g., in terms of depth, number of filters, pooling strategy, batch norm, dropout, data augmentation, etc.). Compare their performance in terms of accuracy and any other relevant measures of classification performance. Provide a brief justification for any choices you make. (Note: Data augmentation in this example could refer to introducing additional digits via rotating, scaling, blurring, ... the existing images or adding your own hand-written digits. It is not necessary, but might be interesting. You should at least think about how data augmentation could relate to the part of your response to Question 7 about risk.)

---

**Question 6.** Choose your operating policy. Explain why you choose the policy you do. At a minimum, report the policy and its   
- manual review rate (≤ 25%),
- auto-processing accuracy,
- incremental profit per 10,000 digits,
- sensitivity to τ (or policy parameters).

---

**Question 7.**  
Prepare a 10-minute presentation for the Head of Ops + Risk/Compliance that explains your system, expected savings, risk controls, limitations, and next steps. You should imagine that your intended audience is open to, but wary of automation.  
